# Hours vs Revenue Analysis

This notebook uses only local exports: HubSpot, Klanten, Facturen, Factuurregels, and Clockify.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
candidates = [Path('outputs'), Path('scripts/wefact_data/outputs')]
base = next((p for p in candidates if (p / 'company_summary.csv').exists()), None)
if base is None:
    raise FileNotFoundError('Could not find company_summary.csv in outputs/ or scripts/wefact_data/outputs/')
company = pd.read_csv(base / 'company_summary.csv')
invoice_lines = pd.read_csv(base / 'invoice_line_matches.csv')
clockify = pd.read_csv(base / 'clockify_time_by_client.csv')
company.head()


In [ ]:
analysis = company.copy()
for col in ['clockify_total_hours', 'invoice_amount_ex_vat', 'invoice_amount_inc_vat', 'revenue_per_hour_ex_vat']:
    analysis[col] = pd.to_numeric(analysis[col], errors='coerce')
analysis = analysis[(analysis['clockify_total_hours'].fillna(0) > 0) | (analysis['invoice_amount_ex_vat'].fillna(0) > 0)].copy()
analysis = analysis.sort_values(['invoice_amount_ex_vat', 'clockify_total_hours'], ascending=[False, False])
analysis[['Company name', 'clockify_total_hours', 'invoice_amount_ex_vat', 'invoice_amount_inc_vat', 'revenue_per_hour_ex_vat']].head(25)


In [ ]:
plot_df = analysis[(analysis['clockify_total_hours'] > 0) & (analysis['invoice_amount_ex_vat'] > 0)].copy()
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(plot_df['clockify_total_hours'], plot_df['invoice_amount_ex_vat'], alpha=0.7, color='#1f6f8b')
ax.set_title('Hours Worked vs Invoiced Revenue (ex VAT)')
ax.set_xlabel('Clockify hours')
ax.set_ylabel('Invoice amount ex VAT')
plt.show()


In [ ]:
product = invoice_lines[invoice_lines['invoice_line_match_status'] == 'matched_to_active_hubspot'].groupby('mapped_clockify_project', dropna=False)['Totaal excl. BTW'].sum().reset_index().sort_values('Totaal excl. BTW', ascending=False)
product.head(20)


In [ ]:
project_hours = clockify[clockify['hubspot_record_id'].notna()].groupby('clockify_project', dropna=False)['clockify_total_hours'].sum().reset_index().sort_values('clockify_total_hours', ascending=False)
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(project_hours['clockify_project'][::-1], project_hours['clockify_total_hours'][::-1], color='#0f766e')
ax.set_title('Hours per Clockify Project')
ax.set_xlabel('Hours')
ax.set_ylabel('Project')
plt.show()
project_hours


In [ ]:
project_revenue = invoice_lines[invoice_lines['invoice_line_match_status'] == 'matched_to_active_hubspot'].groupby('mapped_clockify_project', dropna=False)['Totaal excl. BTW'].sum().reset_index().sort_values('Totaal excl. BTW', ascending=False)
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(project_revenue['mapped_clockify_project'].fillna('Unmapped')[::-1], project_revenue['Totaal excl. BTW'][::-1], color='#b45309')
ax.set_title('Revenue per Mapped Project (ex VAT)')
ax.set_xlabel('Revenue ex VAT')
ax.set_ylabel('Project')
plt.show()
project_revenue


In [ ]:
project_hours_cmp = project_hours.rename(columns={'clockify_project': 'project'})
project_rev_cmp = project_revenue.rename(columns={'mapped_clockify_project': 'project', 'Totaal excl. BTW': 'revenue_ex_vat'})
project_compare = project_hours_cmp.merge(project_rev_cmp, how='outer', on='project').fillna({'clockify_total_hours': 0, 'revenue_ex_vat': 0})
project_compare['revenue_per_hour'] = project_compare['revenue_ex_vat'] / project_compare['clockify_total_hours'].replace({0: pd.NA})
project_compare.sort_values('revenue_ex_vat', ascending=False)


In [ ]:
remaining_unmapped_revenue = invoice_lines[(invoice_lines['invoice_line_match_status'] == 'matched_to_active_hubspot') & (invoice_lines['mapped_clockify_project'].isna())]['Totaal excl. BTW'].sum()
monthly_bucket_revenue = invoice_lines[(invoice_lines['invoice_line_match_status'] == 'matched_to_active_hubspot') & (invoice_lines['mapped_clockify_project'] == 'Monthly / Maandelijks')]['Totaal excl. BTW'].sum()
pd.DataFrame([{'metric': 'monthly_bucket_revenue_ex_vat', 'value': monthly_bucket_revenue}, {'metric': 'remaining_unmapped_revenue_ex_vat', 'value': remaining_unmapped_revenue}])


In [ ]:
project_avg = project_compare[(project_compare['clockify_total_hours'] > 0) & (project_compare['revenue_ex_vat'] > 0)].copy().sort_values('revenue_per_hour', ascending=False)
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(project_avg['project'].fillna('Unmapped')[::-1], project_avg['revenue_per_hour'][::-1], color='#7c3aed')
ax.set_title('Average Revenue per Hour per Project')
ax.set_xlabel('Revenue per hour (ex VAT)')
ax.set_ylabel('Project')
plt.show()
project_avg[['project', 'clockify_total_hours', 'revenue_ex_vat', 'revenue_per_hour']]


In [ ]:
client_avg = analysis[(analysis['clockify_total_hours'] > 0) & (analysis['invoice_amount_ex_vat'] > 0)].copy().sort_values('revenue_per_hour_ex_vat', ascending=False)
fig, ax = plt.subplots(figsize=(12, 8))
top_client_avg = client_avg.head(20)
ax.barh(top_client_avg['Company name'][::-1], top_client_avg['revenue_per_hour_ex_vat'][::-1], color='#2563eb')
ax.set_title('Average Revenue per Hour per Client (Top 20)')
ax.set_xlabel('Revenue per hour (ex VAT)')
ax.set_ylabel('Client')
plt.show()
client_avg[['Company name', 'clockify_total_hours', 'invoice_amount_ex_vat', 'revenue_per_hour_ex_vat']].head(30)


In [ ]:
lowest_client_avg = analysis[(analysis['clockify_total_hours'] > 0) & (analysis['invoice_amount_ex_vat'] > 0) & (analysis['revenue_per_hour_ex_vat'] < 80)].copy().sort_values('revenue_per_hour_ex_vat', ascending=True)
fig, ax = plt.subplots(figsize=(12, max(6, len(lowest_client_avg) * 0.35)))
lowest_client_chart = lowest_client_avg.sort_values('revenue_per_hour_ex_vat', ascending=False)
ax.barh(lowest_client_chart['Company name'], lowest_client_chart['revenue_per_hour_ex_vat'], color='#dc2626')
ax.axvline(80, color='#7f1d1d', linestyle='--', linewidth=1.5)
ax.set_title('Average Revenue per Hour per Client Below EUR 80')
ax.set_xlabel('Revenue per hour (ex VAT)')
ax.set_ylabel('Client')
plt.show()
lowest_client_avg[['Company name', 'clockify_total_hours', 'invoice_amount_ex_vat', 'revenue_per_hour_ex_vat']]


In [ ]:
type_klant = analysis.groupby('Type klant', dropna=False).agg(revenue_ex_vat=('invoice_amount_ex_vat', 'sum'), hours=('clockify_total_hours', 'sum')).reset_index()
type_klant['revenue_per_hour'] = type_klant['revenue_ex_vat'] / type_klant['hours'].replace({0: pd.NA})
type_klant = type_klant.sort_values('revenue_ex_vat', ascending=False)
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(type_klant['Type klant'].fillna('Unknown')[::-1], type_klant['revenue_ex_vat'][::-1], color='#9333ea')
ax.set_title('Revenue by Type klant (ex VAT)')
ax.set_xlabel('Revenue ex VAT')
ax.set_ylabel('Type klant')
plt.show()
type_klant


In [ ]:
type_klant_rph = type_klant[type_klant['hours'] > 0].sort_values('revenue_per_hour', ascending=False)
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(type_klant_rph['Type klant'].fillna('Unknown')[::-1], type_klant_rph['revenue_per_hour'][::-1], color='#7c3aed')
ax.set_title('Revenue per Hour by Type klant')
ax.set_xlabel('Revenue per hour (ex VAT)')
ax.set_ylabel('Type klant')
plt.show()
type_klant_rph[['Type klant', 'hours', 'revenue_ex_vat', 'revenue_per_hour']]


In [ ]:
rechtsvorm = analysis.groupby('Rechtsvorm', dropna=False).agg(revenue_ex_vat=('invoice_amount_ex_vat', 'sum'), hours=('clockify_total_hours', 'sum')).reset_index()
rechtsvorm['revenue_per_hour'] = rechtsvorm['revenue_ex_vat'] / rechtsvorm['hours'].replace({0: pd.NA})
rechtsvorm = rechtsvorm.sort_values('revenue_ex_vat', ascending=False)
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(rechtsvorm['Rechtsvorm'].fillna('Unknown')[::-1], rechtsvorm['revenue_ex_vat'][::-1], color='#0f766e')
ax.set_title('Revenue by Rechtsvorm (ex VAT)')
ax.set_xlabel('Revenue ex VAT')
ax.set_ylabel('Rechtsvorm')
plt.show()
rechtsvorm


In [ ]:
rechtsvorm_rph = rechtsvorm[rechtsvorm['hours'] > 0].sort_values('revenue_per_hour', ascending=False)
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(rechtsvorm_rph['Rechtsvorm'].fillna('Unknown')[::-1], rechtsvorm_rph['revenue_per_hour'][::-1], color='#155e75')
ax.set_title('Revenue per Hour by Rechtsvorm')
ax.set_xlabel('Revenue per hour (ex VAT)')
ax.set_ylabel('Rechtsvorm')
plt.show()
rechtsvorm_rph[['Rechtsvorm', 'hours', 'revenue_ex_vat', 'revenue_per_hour']]


In [ ]:
unmapped_lines = invoice_lines[invoice_lines['mapped_clockify_project'].isna()].copy().sort_values(['invoice_line_match_status', 'Totaal excl. BTW'], ascending=[True, False])
unmapped_lines[['Factuurnr', 'Klantnummer', 'Bedrijfsnaam', 'Productnr', 'Omschrijving', 'Totaal excl. BTW', 'invoice_line_match_status']].head(50)


In [ ]:
outliers = analysis[(analysis['clockify_total_hours'] > 0) & (analysis['revenue_per_hour_ex_vat'] > analysis['revenue_per_hour_ex_vat'].quantile(0.99))].copy()
outliers[['Company name', 'clockify_total_hours', 'invoice_amount_ex_vat', 'revenue_per_hour_ex_vat']].sort_values('revenue_per_hour_ex_vat', ascending=False)
